# Transformação — camada Prata

Processa somente o lote mais recente da Bronze, converte tipos e normaliza nomes para snake_case sem acentos. A Bronze mantém o histórico dos snapshots; a Prata representa o snapshot vigente. Conversões inválidas são registradas em tabelas de qualidade e interrompem o fluxo para evitar perda silenciosa de valores.

In [ ]:
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType, StructField, StructType

spark = SparkSession.builder.getOrCreate()
tabela_bronze = "emendas_parlamentares_bronze"
bronze = spark.table(tabela_bronze)
if "source_batch_id" not in bronze.columns or "source_row_id" not in bronze.columns:
    raise ValueError(
        "A Bronze não possui metadados de lote/linha. Execute a extração atualizada "
        "para carregar um novo snapshot antes da transformação."
    )

ultimo_lote = (
    bronze.where(F.col("source_batch_id").isNotNull())
    .orderBy(F.col("Data_Carga").desc(), F.col("source_batch_id").desc())
    .select("source_batch_id")
    .first()
)
if ultimo_lote is None:
    raise ValueError("Não há lotes novos na tabela Bronze.")

source_batch_id = ultimo_lote["source_batch_id"]
df_origem = bronze.where(F.col("source_batch_id") == source_batch_id)
total_origem = df_origem.count()
if total_origem == 0:
    raise ValueError(f"O lote {source_batch_id} não possui registros.")

In [ ]:
placeholders = ("sem informação", "s/i", "mu")

def texto_limpo(nome_origem):
    valor = F.trim(F.col(nome_origem).cast("string"))
    return F.when(
        F.lower(valor).isin(*placeholders) | (valor == ""),
        F.lit(None).cast("string"),
    ).otherwise(valor)

def formato_brl_valido(nome_origem):
    valor = F.trim(F.col(nome_origem).cast("string"))
    formato_valido = valor.rlike(
        r"^-?(?:\d+|\d{1,3}(?:\.\d{3})+)(?:,\d{1,2})?$"
    )
    normalizado = F.regexp_replace(F.regexp_replace(valor, r"\.", ""), ",", ".")
    digitos_inteiros = F.length(F.regexp_extract(normalizado, r"^-?(\d+)", 1))
    return formato_valido & (digitos_inteiros <= 16)

def valor_brl(nome_origem):
    valor = F.trim(F.col(nome_origem).cast("string"))
    normalizado = F.regexp_replace(F.regexp_replace(valor, r"\.", ""), ",", ".")
    return F.when(
        formato_brl_valido(nome_origem),
        normalizado.cast("decimal(18,2)"),
    ).otherwise(F.lit(None).cast("decimal(18,2)"))

colunas_texto = {
    "codigo_emenda": "Código_da_Emenda",
    "tipo_emenda": "Tipo_de_Emenda",
    "codigo_autor": "Código_do_Autor_da_Emenda",
    "nome_autor": "Nome_do_Autor_da_Emenda",
    "numero_emenda": "Número_da_emenda",
    "localidade_aplicacao": "Localidade_de_aplicação_do_recurso",
    "municipio": "Município",
    "uf": "UF",
    "regiao": "Região",
    "codigo_funcao": "Código_Função",
    "nome_funcao": "Nome_Função",
    "codigo_subfuncao": "Código_Subfunção",
    "nome_subfuncao": "Nome_Subfunção",
    "codigo_programa": "Código_Programa",
    "nome_programa": "Nome_Programa",
    "codigo_acao": "Código_Ação",
    "nome_acao": "Nome_Ação",
    "codigo_plano_orcamentario": "Código_Plano_Orçamentário",
    "nome_plano_orcamentario": "Nome_Plano_Orçamentário",
}

colunas_valor = {
    "valor_empenhado": "Valor_Empenhado",
    "valor_liquidado": "Valor_Liquidado",
    "valor_pago": "Valor_Pago",
    "valor_rap_inscritos": "Valor_Restos_A_Pagar_Inscritos",
    "valor_rap_cancelados": "Valor_Restos_A_Pagar_Cancelados",
    "valor_rap_pagos": "Valor_Restos_A_Pagar_Pagos",
}

df_prata = df_origem.select(
    "source_row_id",
    "source_batch_id",
    "source_file",
    F.col("Data_Carga").alias("data_carga"),
    *[texto_limpo(origem).alias(destino) for destino, origem in colunas_texto.items()],
    F.when(
        F.trim(F.col("Ano_da_Emenda")).rlike(r"^-?\d+$"),
        F.trim(F.col("Ano_da_Emenda")).cast("int"),
    ).otherwise(F.lit(None).cast("int")).alias("ano_emenda"),
    F.when(
        F.trim(F.col("Código_Município_IBGE")).rlike(r"^-\d+$"),
        F.lit(None).cast("string"),
    ).otherwise(texto_limpo("Código_Município_IBGE")).alias("codigo_municipio_ibge"),
    F.when(
        F.trim(F.col("Código_UF_IBGE")).rlike(r"^\d+$"),
        F.trim(F.col("Código_UF_IBGE")).cast("int"),
    ).otherwise(F.lit(None).cast("int")).alias("codigo_uf_ibge"),
    *[valor_brl(origem).alias(destino) for destino, origem in colunas_valor.items()],
)

campos_rejeitados = []
for destino, origem in colunas_valor.items():
    valor_origem = F.trim(F.col(origem).cast("string"))
    invalido = (
        F.col(origem).isNotNull()
        & (valor_origem != "")
        & ~formato_brl_valido(origem)
    )
    campos_rejeitados.append(
        df_origem.where(invalido).select(
            "source_row_id",
            "source_batch_id",
            F.lit(destino).alias("campo"),
            F.col(origem).cast("string").alias("valor_original"),
        )
    )

ano_origem = F.trim(F.col("Ano_da_Emenda"))
campos_rejeitados.append(
    df_origem.where(
        F.col("Ano_da_Emenda").isNotNull()
        & (ano_origem != "")
        & ~ano_origem.rlike(r"^-?\d+$")
    ).select(
        "source_row_id", "source_batch_id",
        F.lit("ano_emenda").alias("campo"),
        F.col("Ano_da_Emenda").cast("string").alias("valor_original"),
    )
)

uf_origem = F.trim(F.col("Código_UF_IBGE"))
campos_rejeitados.append(
    df_origem.where(
        F.col("Código_UF_IBGE").isNotNull()
        & (uf_origem != "")
        & ~uf_origem.rlike(r"^\d+$")
        & ~uf_origem.rlike(r"^-\d+$")
    ).select(
        "source_row_id", "source_batch_id",
        F.lit("codigo_uf_ibge").alias("campo"),
        F.col("Código_UF_IBGE").cast("string").alias("valor_original"),
    )
)

df_rejeicoes = reduce(lambda left, right: left.unionByName(right), campos_rejeitados)
total_rejeicoes = df_rejeicoes.count()
total_sentinelas_normalizadas = (
    df_origem.where(F.trim(F.col("Código_UF_IBGE")).rlike(r"^-\d+$")).count()
    + df_origem.where(F.trim(F.col("Código_Município_IBGE")).rlike(r"^-\d+$")).count()
)

schema_metricas = StructType([
    StructField("source_batch_id", StringType(), False),
    StructField("total_registros_origem", LongType(), False),
    StructField("total_conversoes_invalidas", LongType(), False),
    StructField("total_sentinelas_normalizadas", LongType(), False),
])
df_metricas = spark.createDataFrame(
    [(source_batch_id, total_origem, total_rejeicoes, total_sentinelas_normalizadas)],
    schema_metricas,
    ).withColumn("data_verificacao", F.current_timestamp())

(
    df_rejeicoes.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("emendas_parlamentares_rejeicoes")
)
(
    df_metricas.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("emendas_parlamentares_quality_metrics")
)

tabela_prata = "emendas_parlamentares_prata"
(
    df_prata.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabela_prata)
)

total_prata = spark.table(tabela_prata).count()
ids_nulos = spark.table(tabela_prata).where(F.col("source_row_id").isNull()).count()
ids_duplicados = (
    spark.table(tabela_prata).groupBy("source_row_id").count()
    .where(F.col("count") > 1).count()
)
print(f"Lote: {source_batch_id}")
print(f"Linhas Bronze: {total_origem}; linhas Prata: {total_prata}")
print(f"Conversões inválidas: {total_rejeicoes}")
print(f"Códigos sentinela normalizados para nulo: {total_sentinelas_normalizadas}")
if total_prata != total_origem or ids_nulos or ids_duplicados:
    raise ValueError("Falha na reconciliação ou unicidade da chave técnica da Prata.")
if total_rejeicoes:
    df_rejeicoes.groupBy("campo").count().orderBy(F.desc("count")).show()
    df_rejeicoes.show(20, truncate=False)
    raise ValueError(
        "Há conversões inválidas. Consulte emendas_parlamentares_rejeicoes "
        "e corrija/valide os dados antes de promover a Gold."
    )